# Loss Functions

In Lesson 04, we built a Multi-Layer Perceptron and executed Forward Propagation. We fed data into the network, multiplied it by random weights, and generated a prediction ($\hat{y}$).

Because the initial weights are entirely random, that first prediction is guaranteed to be completely wrong. Before the network can learn and improve, it must accurately measure *how wrong* it is. It needs a mathematical ruler.

This ruler is the **Loss Function** (or Cost Function). It takes the network's prediction ($\hat{y}$) and compares it to the ground-truth reality ($y$), outputting a single scalar value representing the total error. The entire goal of Deep Learning is to drive this number as close to zero as possible.

Let's set up our PyTorch environment to mathematically measure failure.

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch Loss Functions Environment Ready.")

✅ PyTorch Loss Functions Environment Ready.


# 1. Regression Loss: Mean Squared Error (MSE)

When your Neural Network is predicting a continuous, infinite number (e.g., the price of a house, tomorrow's temperature), we use **Mean Squared Error**.

$$L_{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

* $y_i$: The true value.
* $\hat{y}_i$: The network's prediction.
* **Why square it?** Two reasons. First, squaring removes negative signs (guessing $-\$100$ is just as bad as guessing $+\$100$). Second, it creates a parabola. A quadratic curve mathematically punishes massive outliers exponentially harder than tiny mistakes, pushing the network to avoid catastrophic failures.

# 2. Binary Classification Loss: Binary Cross-Entropy (BCE)

When your network is predicting a Yes/No outcome (e.g., Is this transaction fraudulent? $1$ for Yes, $0$ for No), MSE is the wrong tool.

If the true answer is $1$, and your network uses a Sigmoid activation to predict $0.99$, the error should be tiny. If it predicts $0.01$, the error should be massive. We measure this using logarithms in a function called **Binary Cross-Entropy (Log Loss)**.

$$L_{BCE} = - \frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

### The Mathematics of Model Overconfidence

Look closely at the equation.

* If the true answer is $y = 1$, the right half of the equation $(1 - 1)$ becomes $0$ and disappears. We are left with $- \log(\hat{y})$.
* If the network predicts $\hat{y} = 0.99$, $-\log(0.99)$ is practically $0$. Perfect!
* If the network confidently predicts $\hat{y} = 0.0001$, $-\log(0.0001)$ explodes toward infinity. The network is punished violently for being confidently wrong.

# 3. Multi-Class Classification Loss: Categorical Cross-Entropy (CCE)

If you are predicting more than two categories (e.g., Is this image a Cat, Dog, or Bird?), you cannot use a single Sigmoid neuron. You must use multiple output neurons and apply a **Softmax** activation.

Softmax forces the outputs of all neurons to sum exactly to $1.0$, creating a mathematical probability distribution.
Once you have Softmax probabilities, you measure the error against the true category using **Categorical Cross-Entropy**.

$$L_{CCE} = - \sum_{c=1}^{C} y_c \log(\hat{y}_c)$$


*(Where $y_c$ is a One-Hot Encoded vector. If the image is a Dog (class 2 of 3), the true vector is $[0, 1, 0]$.)*

# 4. The PyTorch Engineering Trap: The Logits Trick

There is a massive engineering "gotcha" in PyTorch that traps almost all beginners.

Mathematically, a classification network requires an activation function (Sigmoid or Softmax) at the end, followed by the Loss calculation (BCE or CCE).
If you write this literally in PyTorch (`nn.Sigmoid()` $\rightarrow$ `nn.BCELoss()`), your network will suffer from **Numerical Instability**. Calculating the exponential $e^{-x}$ for Sigmoid, and then immediately calculating the logarithm $\log(x)$ for the Loss, causes computers to hit floating-point rounding errors when numbers get too close to zero.

**The Enterprise Standard**: You must drop the final activation function from your network. You output the raw, unactivated numbers (called **Logits**). You then use special PyTorch loss functions (`BCEWithLogitsLoss` and `CrossEntropyLoss`) that combine the activation and the logarithm into a single, mathematically stable, highly-optimized C++ operation.

In [2]:
# 1. Simulating Raw Network Outputs (Logits)
# Imagine a network with 1 output neuron predicting Fraud on 3 transactions.
# These are RAW outputs. No Sigmoid has been applied yet.
logits_binary = torch.tensor([[-2.5], [0.1], [3.4]], dtype=torch.float32)

# The ground-truth reality (0 = Safe, 1 = Fraud)
y_true_binary = torch.tensor([[0.0], [0.0], [1.0]], dtype=torch.float32)

# 2. The WRONG Way (Mathematically correct, computationally unstable)
sigmoid = nn.Sigmoid()
loss_fn_wrong = nn.BCELoss()
probs = sigmoid(logits_binary)
loss_wrong = loss_fn_wrong(probs, y_true_binary)

# 3. The RIGHT Way (Enterprise PyTorch Standard)
# Notice we pass the RAW logits directly into the loss function!
loss_fn_right = nn.BCEWithLogitsLoss()
loss_right = loss_fn_right(logits_binary, y_true_binary)

print("--- Binary Classification Loss ---")
print(f"Loss calculated the unstable way: {loss_wrong.item():.4f}")
print(f"Loss calculated the stable way:   {loss_right.item():.4f}\n")


# 4. Multi-Class Classification (e.g., Cat=0, Dog=1, Bird=2)
# Imagine 2 images passing through 3 output neurons (RAW Logits)
logits_multi = torch.tensor([
    [ 2.0, -1.0,  0.5], # Image 1: Network strongly suspects class 0 (Cat)
    [-0.5,  3.0, -1.0]  # Image 2: Network strongly suspects class 1 (Dog)
], dtype=torch.float32)

# Ground truth: Image 1 is a Cat (0), Image 2 is a Bird (2)
# Notice PyTorch expects the class INDEX, not a one-hot vector!
y_true_multi = torch.tensor([0, 2], dtype=torch.long)

# CrossEntropyLoss automatically applies Softmax AND calculates the negative log-likelihood
loss_fn_multi = nn.CrossEntropyLoss()
loss_multi = loss_fn_multi(logits_multi, y_true_multi)

print("--- Multi-Class Classification Loss ---")
print(f"Loss for the Batch: {loss_multi.item():.4f}")
print("Insight: The loss is high because the network was horribly wrong about Image 2! It confidently guessed Dog (3.0), but the truth was Bird.")

--- Binary Classification Loss ---
Loss calculated the unstable way: 0.2854
Loss calculated the stable way:   0.2854

--- Multi-Class Classification Loss ---
Loss for the Batch: 2.1443
Insight: The loss is high because the network was horribly wrong about Image 2! It confidently guessed Dog (3.0), but the truth was Bird.


## Real-World Use Case or Analogy:

Think of the Loss Function like a **Golf Coach evaluating your swing**:

* **Mean Squared Error (The Precision Coach)**: You are trying to hit the ball exactly 100 yards. If you hit it 99 yards, the coach says, "Minus 1 point." If you hit it 150 yards, the coach screams, "MINUS 2,500 POINTS!" (Squaring the error). The coach forces you to avoid massive, wild swings.
* **Binary Cross-Entropy (The Confidence Coach)**: The coach asks, "Will this putt go in?" You answer, "I am 99% sure it will."
* If the putt goes in, the coach is happy. (Loss = 0).
* If the putt misses, the coach completely loses their mind. You weren't just wrong; you were *arrogantly* wrong. The punishment (Loss) approaches infinity. If you had said "I am only 51% sure," the punishment for missing would have been much smaller. BCE trains the network to calibrate its confidence properly!